# Fine-tune Qwen3.5 0.8B to reason briefly (AQuA-RAT, LoRA)

Runtime: **GPU, L4 or A100** (Runtime > Change runtime type). Then Runtime > Run all.

If the session dies, run all again: the data is kept in Drive and training resumes from the last checkpoint.
Only AQuA-RAT is used here. M3Kang never leaves the laptop.

In [ ]:
# The only settings to edit
REPO_URL = "https://github.com/marcsanchezsala/datathon-UPC-maths-qualcomm-arduino-unoq"
REPO_BRANCH = "bigger-model"   # the branch with finetune/ pushed
DRIVE_DIR = "/content/drive/MyDrive/m3-finetune"
N_EXAMPLES = 15000
EPOCHS = 2
BATCH, ACCUM = 8, 2            # 16 examples per step; use BATCH, ACCUM = 16, 1 on an A100

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!rm -rf /content/repo && git clone -q --depth 1 -b {REPO_BRANCH} {REPO_URL} /content/repo
%cd /content/repo/finetune
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
# 1. Data (about 2 minutes, done once)
DATA = f"{DRIVE_DIR}/aqua.jsonl"
if not os.path.exists(DATA):
    !python prepare_aqua.py --out {DATA} --n {N_EXAMPLES}
# The training prompt must be the copilot's inference prompt: this fails if they differ
!python check_format.py --data {DATA}

In [ ]:
# 2. Training (resumes from the last checkpoint in Drive)
RUN = f"{DRIVE_DIR}/run"
!python train.py --data {DATA} --out {RUN} --epochs {EPOCHS} --batch {BATCH} --accum {ACCUM}

In [ ]:
# 3. Merge, convert with llama.cpp at the board's commit, Q8_0, SHA256
GGUF = f"{RUN}/Qwen3.5-0.8B-aqua-reason-Q8_0.gguf"
!python export.py --adapter {RUN}/adapter --out {GGUF} --llama-cpp /content/llama.cpp --work /content/merged
!ls -la {RUN}